In [1]:
# =====================================================================
# CELL 1 — Install + Mount Drive
# =====================================================================
!pip install -U "transformers>=4.48.0" accelerate bitsandbytes qwen-vl-utils datasets tqdm -q

import os
from google.colab import drive
drive.mount('/content/drive')

# Model weights to temp disk (avoids storage quota errors on your account)
os.environ["HF_HOME"] = "/content/model_cache"
os.environ["HF_DATASETS_CACHE"] = "/content/dataset_cache"

# RESULTS_ROOT is pointed to the shared folder on your Drive
RESULTS_ROOT = "/content/drive/MyDrive/GSV_Math_Model_Cache/gsv_math_results"
os.makedirs(RESULTS_ROOT, exist_ok=True)

import torch
print(f"\nCUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 70.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 17.8 MB/s eta 0:00:00
Mounted at /content/drive

CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


In [2]:
# =====================================================================
# CELL 2 — Improved Parsing Logic v2 (tail-biased extraction)
# =====================================================================
import re

FINAL_ANSWER_PATTERNS = [
    r'\\boxed\{([^}]*)\}',
    r'[Ff]inal\s*[Aa]nswer\s*[:\-]?\s*(.{1,80})',
    r'[Tt]herefore[,\s]+(?:the\s+)?(?:answer|value|result)\s+is\s*[:\-]?\s*(.{1,80})',
    r'[Tt]he\s+answer\s+is\s*[:\-]?\s*(.{1,80})',
    r'[Ss]o\s+the\s+answer\s+is\s*[:\-]?\s*(.{1,80})',
    r'=\s*(\S+)\s*$',
]

def extract_final_answer_region(raw_text, tail_chars=300):
    for pattern in FINAL_ANSWER_PATTERNS:
        matches = list(re.finditer(pattern, raw_text, re.IGNORECASE | re.DOTALL))
        if matches:
            return matches[-1].group(1).strip()
    return raw_text[-tail_chars:] if len(raw_text) > tail_chars else raw_text

def clean_free_form(text):
    if not isinstance(text, str): return str(text)
    text = text.strip().lower()
    prefixes = ["the answer is", "therefore, the answer is", "so the answer is",
                "the value is", "answer is", "value is", "equals", "it is",
                "the final answer is", "final answer:", "answer:"]
    for prefix in prefixes:
        if text.startswith(prefix):
            text = text[len(prefix):].strip()
    match = re.match(r'^[a-zA-Z\s]+=\s*(.*)$', text)
    if match: text = match.group(1).strip()
    return text.rstrip('.!?*, ')

def get_most_similar(extraction, choices):
    distances = [-len(set(extraction.lower()) & set(choice.lower())) for choice in choices]
    return choices[distances.index(min(distances))]

def normalize_extracted_answer(extraction, choices, question_type, answer_type):
    extraction = str(extraction).strip() if extraction else ""
    extraction = extract_final_answer_region(extraction)

    if question_type == 'multi_choice':
        letter = re.findall(r'\(([a-zA-Z])\)', extraction)
        extraction = letter[0].upper() if letter else extraction
        options = [chr(ord('A') + i) for i in range(len(choices))]
        if extraction in options:
            extraction = choices[options.index(extraction)]
        else:
            extraction = get_most_similar(clean_free_form(extraction), choices)
    else:
        cleaned = clean_free_form(extraction)
        if answer_type in ['integer', 'float']:
            numbers = re.findall(r'-?\d+\.?\d*', cleaned)
            extraction = numbers[-1] if numbers else cleaned
    return extraction

def is_correct(pred, gt, answer_type):
    if pred.lower().strip() == gt.lower().strip(): return 1
    if answer_type in ['integer', 'float']:
        try:
            if abs(float(pred) - float(gt)) < 1e-5: return 1
        except: pass
    return 0

In [3]:
# =====================================================================
# CELL 3 — Config (Self-Consistency k=5)
# =====================================================================
MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
MAX_NEW_TOKENS = 512
NUM_PATHS = 5  # Number of generations per question

RUN_NAME = "qwen25vl7b_selfconsistency_k5"
BATCH_SIZE = 20  # Lower batch size so it checkpoints frequently

RUN_DIR = f"{RESULTS_ROOT}/{RUN_NAME}"
import os
os.makedirs(RUN_DIR, exist_ok=True)

print(f"Target Model:           {MODEL_ID}")
print(f"Self-Consistency Paths: k={NUM_PATHS}")
print(f"Run name:               {RUN_NAME}")
print(f"Checkpoints path:       {RUN_DIR}")

Target Model:           Qwen/Qwen2.5-VL-7B-Instruct
Self-Consistency Paths: k=5
Run name:               qwen25vl7b_selfconsistency_k5
Checkpoints path:       /content/drive/MyDrive/GSV_Math_Model_Cache/gsv_math_results/qwen25vl7b_selfconsistency_k5


In [4]:
# =====================================================================
# CELL 4 — Load Dataset and Model (4-bit)
# =====================================================================
from datasets import load_dataset
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
import torch

print("Loading MathVista testmini dataset...")
mathvista = load_dataset("AI4Math/MathVista", split="testmini")
print(f"Loaded {len(mathvista)} test samples.")

print(f"\nLoading model: {MODEL_ID} in 4-bit...")
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto"
)
processor = AutoProcessor.from_pretrained(MODEL_ID)
print("Model and Processor loaded successfully.")

Loading MathVista testmini dataset...


README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

data/testmini-00000-of-00001-725687bf7a1(…): reconstructing file:   0%|          |  0.00B /  142MB            

data/testmini-00000-of-00001-725687bf7a1(…): downloading bytes:           |  0.00B            

data/test-00000-of-00002-6b81bd7f7e2065e(…): reconstructing file:   0%|          |  0.00B /  358MB            

...

tokenizer_config.json:   0%|          | 0.00/5.70k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Model and Processor loaded successfully.


In [5]:
# =====================================================================
# CELL 5 — Self-Consistency Loop (with parallel/sequential generation)
# =====================================================================
import gc, json
from pathlib import Path
from tqdm import tqdm
from collections import Counter
from qwen_vl_utils import process_vision_info
import torch

RESOLUTION_LADDER = [1003520, 501760, 313600]

def load_completed_pids():
    done = set()
    for f in Path(RUN_DIR).glob("batch_*.json"):
        try:
            with open(f) as fh:
                batch = json.load(fh)
            done.update(item["question_id"] for item in batch)
        except json.JSONDecodeError:
            print(f"  [warn] {f.name} looked incomplete, ignoring.")
    return done

def next_batch_index():
    existing = list(Path(RUN_DIR).glob("batch_*.json"))
    if not existing: return 0
    return max(int(f.stem.split("_")[1]) for f in existing) + 1

def generate_k_paths(sample, max_px):
    """Attempt parallel path generation. If OOM, fall back to sequential."""
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": sample["decoded_image"], "max_pixels": max_px},
                {"type": "text", "text": sample["query"]}
            ],
        }
    ]
    text_prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(text=[text_prompt], images=image_inputs, padding=True, return_tensors="pt").to(model.device)

    raw_answers = []
    try:
        # STRATEGY 1: Generate all 5 paths in a single batched step (Fastest)
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=True,
                temperature=0.7,
                num_return_sequences=NUM_PATHS
            )
        for j in range(NUM_PATHS):
            gen_ids = output_ids[j][len(inputs.input_ids[0]):]
            decoded = processor.decode(gen_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)
            raw_answers.append(decoded)

        del inputs, output_ids
    except torch.cuda.OutOfMemoryError:
        # STRATEGY 2: Fallback to generating 1-by-1 sequentially (Safe against VRAM limits)
        print(f"\n  [VRAM warning] Batched k={NUM_PATHS} OOM'd at resolution {max_px}. Falling back to sequential...")
        gc.collect()
        torch.cuda.empty_cache()

        raw_answers = []
        for path_idx in range(NUM_PATHS):
            with torch.no_grad():
                output_ids = model.generate(
                    **inputs,
                    max_new_tokens=MAX_NEW_TOKENS,
                    do_sample=True,
                    temperature=0.7,
                    num_return_sequences=1
                )
            gen_ids = output_ids[0][len(inputs.input_ids[0]):]
            decoded = processor.decode(gen_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)
            raw_answers.append(decoded)
            del output_ids
            gc.collect()
            torch.cuda.empty_cache()

        del inputs

    gc.collect()
    torch.cuda.empty_cache()
    return raw_answers

def run_self_consistency(sample):
    """Progressive resolution wrapper for self-consistency."""
    for max_px in RESOLUTION_LADDER:
        try:
            raw_answers = generate_k_paths(sample, max_px)
            return raw_answers
        except torch.cuda.OutOfMemoryError:
            gc.collect()
            torch.cuda.empty_cache()
            if max_px == RESOLUTION_LADDER[-1]:
                return ["[OOM_SKIP]"] * NUM_PATHS
            continue
    return ["[OOM_SKIP]"] * NUM_PATHS

# --- Main Loop ---
completed_pids = load_completed_pids()
print(f"Resuming run '{RUN_NAME}': {len(completed_pids)}/{len(mathvista)} samples already done.")

remaining = [s for s in mathvista if s["pid"] not in completed_pids]
print(f"{len(remaining)} samples left to run.\n")

batch_idx = next_batch_index()
batch_results = []

for i, sample in enumerate(tqdm(remaining, desc="Self-Consistency (k=5)")):
    raw_answers = run_self_consistency(sample)

    if "[OOM_SKIP]" in raw_answers:
        final_pred = "[OOM]"
        correct_flag = 0
    else:
        # Extract and normalize answers from all 5 paths
        predictions = []
        for raw_ans in raw_answers:
            pred = normalize_extracted_answer(raw_ans, sample.get("choices", []), sample["question_type"], sample["answer_type"])
            predictions.append(pred)

        # Majority Vote
        votes = Counter(predictions)
        final_pred = votes.most_common(1)[0][0] # Select the answer with highest vote count
        correct_flag = is_correct(final_pred, sample["answer"], sample["answer_type"])

    batch_results.append({
        "question_id": sample["pid"],
        "skills": sample["metadata"]["skills"],
        "correct": correct_flag,
        "final_prediction": final_pred,
        "all_predictions": predictions if "[OOM_SKIP]" not in raw_answers else [],
        "raw_answers": raw_answers
    })

    if len(batch_results) >= BATCH_SIZE or i == len(remaining) - 1:
        out_file = f"{RUN_DIR}/batch_{batch_idx:04d}.json"
        with open(out_file, "w") as fh:
            json.dump(batch_results, fh)
        print(f"  [checkpoint] saved {out_file}  ({len(batch_results)} samples)")
        batch_idx += 1
        batch_results = []

print("\nBatch complete. Run this cell again to resume if needed.")

Resuming run 'qwen25vl7b_selfconsistency_k5': 980/1000 samples already done.
20 samples left to run.





Self-Consistency (k=5):   0%|          | 0/20 [00:00<?, ?it/s]

Self-Consistency (k=5):   5%|▌         | 1/20 [01:31<29:05, 91.89s/it]

Self-Consistency (k=5):  10%|█         | 2/20 [03:05<27:51, 92.88s/it]

Self-Consistency (k=5):  15%|█▌        | 3/20 [04:48<27:38, 97.54s/it]

Self-Consistency (k=5):  20%|██        | 4/20 [06:18<25:13, 94.61s/it]

Self-Consistency (k=5):  25%|██▌       | 5/20 [07:50<23:26, 93.78s/it]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:957: UserWarning: inner dimension (3420) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(


Self-Consistency (k=5):  30%|███       | 6/20 [10:33<27:19, 117.08s/it]

Self-Consistency (k=5):  35%|███▌      | 7/20 [13:49<30:57, 142.88s/it]

Self-Consistency (k=5):  40%|████      | 8/20 [14:48<23:13, 116.11s/it]

Self-Consistency (k=5):  45%|████▌     | 9/20 [16:02<18:55, 103.22s/it]

Self-Consistency (k=5):  50%|█████     | 10/20 [18:09<18:23, 110.39s/i

  [checkpoint] saved /content/drive/MyDrive/GSV_Math_Model_Cache/gsv_math_results/qwen25vl7b_selfconsistency_k5/batch_0049.json  (20 samples)

Batch complete. Run this cell again to resume if needed.


In [6]:
# =====================================================================
# CELL 6 — Aggregate all batches
# =====================================================================
import json, glob

results = []
for f in sorted(glob.glob(f"{RUN_DIR}/batch_*.json")):
    with open(f) as fh:
        results.extend(json.load(fh))

print(f"Aggregated {len(results)} / {len(mathvista)} total samples")

if len(results) >= len(mathvista):
    final_path = f"{RUN_DIR}/FINAL_results.json"
    with open(final_path, "w") as fh:
        json.dump(results, fh)
    print(f"Saved: {final_path}")
else:
    print(f"\nNot complete yet. Re-run Cell 5 to continue.")

Aggregated 1000 / 1000 total samples
Saved: /content/drive/MyDrive/GSV_Math_Model_Cache/gsv_math_results/qwen25vl7b_selfconsistency_k5/FINAL_results.json


In [7]:
# =====================================================================
# CELL 7 — Metrics + Comparison
# =====================================================================
skill_to_category = {
    "geometry reasoning": "geometry",
    "arithmetic reasoning": "arithmetic",
    "algebraic reasoning": "algebra",
    "logical reasoning": "logic",
    "numeric commonsense": "numeric",
    "scientific reasoning": "scientific",
    "statistical reasoning": "statistical",
}

categories = ["all", "geometry", "arithmetic", "algebra", "logic", "numeric", "scientific", "statistical"]
metrics = {cat: {"correct": 0, "total": 0} for cat in categories}

for res in results:
    correct = res["correct"]
    metrics["all"]["correct"] += correct
    metrics["all"]["total"] += 1
    for skill in res.get("skills", []):
        cat = skill_to_category.get(skill)
        if cat in metrics:
            metrics[cat]["correct"] += correct
            metrics[cat]["total"] += 1

print("\n" + "="*60)
print(f"{'Qwen2.5-VL-7B — Self-Consistency (k=5)':^60}")
print("="*60)
print(f"{'Category':<20} | {'Correct':<10} | {'Total':<10} | {'Accuracy':<10}")
print("-"*60)

for cat in categories:
    correct = metrics[cat]["correct"]
    total = metrics[cat]["total"]
    acc = (correct / total * 100) if total > 0 else 0.0
    print(f"{cat.capitalize():<20} | {correct:<10} | {total:<10} | {acc:.2f}%")

print("="*60)

# Compare pipeline stages
print(f"\n{'PIPELINE ABLATION COMPARISON':^60}")
print("-"*60)
sc_acc = metrics['all']['correct'] / metrics['all']['total'] * 100
print(f"  Stage 1: Zero-shot (v2 parser):  61.9%")
print(f"  Stage 2: Few-shot (v2 parser):   62.1%")
print(f"  Stage 3: Self-Consistency (k=5): {sc_acc:.1f}%")
print(f"  Current Delta (vs Zero-shot):    {sc_acc - 61.9:+.1f}%")
print("="*60)


           Qwen2.5-VL-7B — Self-Consistency (k=5)           
Category             | Correct    | Total      | Accuracy  
------------------------------------------------------------
All                  | 664        | 1000       | 66.40%
Geometry             | 152        | 239        | 63.60%
Arithmetic           | 224        | 353        | 63.46%
Algebra              | 176        | 281        | 62.63%
Logic                | 12         | 37         | 32.43%
Numeric              | 61         | 144        | 42.36%
Scientific           | 70         | 122        | 57.38%
Statistical          | 239        | 301        | 79.40%

                PIPELINE ABLATION COMPARISON                
------------------------------------------------------------
  Stage 1: Zero-shot (v2 parser):  61.9%
  Stage 2: Few-shot (v2 parser):   62.1%
  Stage 3: Self-Consistency (k=5): 66.4%
  Current Delta (vs Zero-shot):    +4.5%
